What days to BTS, SFDPS and Eurocontrol have in common. So I can get the smallest set of raw ADS-B data possible. 

WHAT SFDPS days do I actually have? and what are the trends in them ?
Maybe I could download all the SFDPS data to my desktop and then I could delete it.  I need to calculate how much that would cost thought.... 

How to cheapely query what days of s3 data we have? 

Eurocontorl would be next easiset. 

# Eurocontrol sets

In [1]:
from pathlib import Path
import re

folder = Path("/Volumes/T2-SSD/planequery/data/raw/eurocontrol")

months = sorted(
    path.name
    for path in folder.iterdir()
    if path.is_dir() and re.fullmatch(r"\d{6}", path.name)
)

if months:
    print(f"Range: {months[0]} to {months[-1]}")
else:
    print("No YYYYMM folders found.")

Range: 201501 to 202509


In [ ]:
START_DATE = date(2024, 9, 1)
END_DATE = date(2025, 9, 1)  # Inclusive
# download adsblol and adsbx for this. algorithm segmentation

SFDPS...

In [ ]:
DATES = [date(2026,2,1), date(2026,3,1)] # SFDPS AND BTS can use this

In [ ]:
EUROCONTROL_TEST_DATES = [date(2025,4,1), date(2025,5,1), date(2025,6,1)]

In [4]:
from dataclasses import dataclass
from datetime import date
from typing import Any, Mapping

import polars as pl

from data_engineering.adsb.read_adsb import ADSBSource
from flights.flight_source import FlightSource
from flights.sources import FLIGHT_SOURCES_BY_NAME


@dataclass(frozen=True, slots=True)
class EvaluationStats:
    evaluation_date: date
    adsb_source: ADSBSource | None
    algorithm_source: FlightSource
    gold_source: FlightSource

    use_all_icaos: bool
    filter_rotorcraft: bool
    pia_or_american_ladd_only: bool

    true_positive: int
    false_positive: int
    false_negative: int

    takeoff_airport_match_count: int
    landing_airport_match_count: int
    airport_match_count: int

    @property
    def precision(self) -> float:
        denominator = self.true_positive + self.false_positive
        return self.true_positive / denominator if denominator else 0.0

    @property
    def recall(self) -> float:
        denominator = self.true_positive + self.false_negative
        return self.true_positive / denominator if denominator else 0.0

    @property
    def f1(self) -> float:
        denominator = self.precision + self.recall
        return (
            2 * self.precision * self.recall / denominator
            if denominator
            else 0.0
        )

    @property
    def takeoff_airport_match_pct(self) -> float:
        return self._airport_percentage(self.takeoff_airport_match_count)

    @property
    def landing_airport_match_pct(self) -> float:
        return self._airport_percentage(self.landing_airport_match_count)

    @property
    def airport_match_pct(self) -> float:
        return self._airport_percentage(self.airport_match_count)

    def _airport_percentage(self, match_count: int) -> float:
        return (
            match_count / self.true_positive
            if self.true_positive
            else 0.0
        )

    @classmethod
    def from_csv_row(
        cls,
        row: Mapping[str, Any],
    ) -> "EvaluationStats":
        if row["test_date"] == "average":
            raise ValueError(
                "Average rows do not contain actual integer counts"
            )

        return cls(
            evaluation_date=date.fromisoformat(row["test_date"]),
            adsb_source=parse_optional_adsb_source(row["adsb_src"]),
            algorithm_source=resolve_flight_source(row["test_src"]),
            gold_source=resolve_flight_source(row["gold_src"]),
            use_all_icaos=bool(row["use_all_icaos"]),
            filter_rotorcraft=bool(row["filter_rotorcraft"]),
            pia_or_american_ladd_only=bool(
                row["pia_or_american_ladd_only"]
            ),
            true_positive=parse_count(row["true_positive"]),
            false_positive=parse_count(row["false_positive"]),
            false_negative=parse_count(row["false_negative"]),
            takeoff_airport_match_count=parse_count(
                row["takeoff_airport_id_match_count"]
            ),
            landing_airport_match_count=parse_count(
                row["landing_airport_id_match_count"]
            ),
            airport_match_count=parse_count(
                row["airport_id_match_count"]
            ),
        )

def parse_count(value: int | float) -> int:
    """Convert CSV values such as 12457.0 into validated integer counts."""
    number = float(value)

    if not number.is_integer():
        raise ValueError(f"Expected an integer count, got {value!r}")

    return int(number)


def parse_optional_adsb_source(
    value: str | None,
) -> ADSBSource | None:
    if value is None or not value.strip():
        return None

    return ADSBSource(value)


def resolve_flight_source(name: str) -> FlightSource:
    try:
        return FLIGHT_SOURCES_BY_NAME[name]
    except KeyError:
        raise ValueError(f"Unknown flight source: {name!r}") from None

# Data Analysis

In [1]:
bts_csv_path = "/Volumes/T2-SSD/planequery/data/experiments/comparison/2026-10-03_20-56-20-039148_18884de_flights_evaluation.csv" 
sfpds_csv_path = "/Volumes/T2-SSD/planequery/data/experiments/comparison/2026-10-04_19-02-38-491861_18884de_flights_evaluation.csv"
# eurocontrol_csv_path = 

In [5]:
import polars as pl
df = pl.read_csv(bts_csv_path)
bts_stats = [
    EvaluationStats.from_csv_row(row)
    for row in df.filter(
        pl.col("test_date") != "average"
    ).to_dicts()
]

In [8]:
bts_stats[0].evaluation_date

datetime.date(2026, 2, 1)

In [ ]:
# we could have class of stats